In [2]:
import tensorflow as tf
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from tensorflow.keras import optimizers

In [ ]:
def validity_reward(mol):
    if mol is None:
      return 0
    # Sanitize
    try:
      Chem.SanitizeMol(mol)
    except Exception:
      return 0
    # Check Kekulization
    try:
      Chem.Kekulize(mol, clearAromaticFlags = True)
    except Exception:
      return 0
    # Check Valency
    for atom in mol.GetAtoms():
      explicit_valence = atom.GetExplicitValence()
      if explicit_valence > atom.GetTotalValence():
        return 0
    return 2

In [5]:
def morg_fp(mol):
  fp_gen = rdFingerprintGenerator.GetMorganGenerator(
                                                    radius = 2,
                                                    fpSize = 2048
                                                    )
  return fp_gen.GetFingerprint(mol)

In [ ]:
def uniqueness_reward(gen_smiles, curr_mol):
  curr_smiles = Chem.MolToSmiles(curr_mol)
  if validity_reward(curr_mol) == 0:
      return 0
  if curr_smiles in set(gen_smiles):
      return 0
  else:
      return 1

In [ ]:
def novelty_reward(curr_mol, train_smiles, generated_smiles):
    unique_train = set(train_smiles)
    if validity_reward(curr_mol) == 0:
        return 0
    elif uniqueness_reward(generated_smiles, curr_mol) == 0:
        return 0
    curr_smiles = Chem.MolToSmiles(curr_mol)
    if curr_smiles in unique_train:
        return 0
    else:
        return 3

In [8]:
def reinforce_train(generator, dataset, num_epoch = 1000):
    optimizer = optimizers.Adam(learning_rate = 0.001)
    generated_mols = []
    for epoch in range(num_epoch):
        with tf.GradientTape() as tape:
            z = tf.random.normal((1, 16))  
            adj_matrix, node_features = generator(z) 
            atom_types = set(adj_matrix[1])
            mol = adjacency_matrix_to_mol(adj_matrix, atom_types)
            generated_mols.append(mol)

            validity = validity_reward(mol)
            uniqueness = uniqueness_reward(mol, generated_mols)
            novelty = novelty_reward(mol, train_mols, generated_mols)

            total_reward = validity + uniqueness + novelty
            
            # Compute loss as negative reward (maximize reward)
            loss = -total_reward  

        # Compute gradients and update generator
        gradients = tape.gradient(loss, generator.trainable_variables)
        optimizer.apply_gradients(zip(gradients, generator.trainable_variables))

        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Reward: {total_reward.numpy()}")
